In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from datetime import date, timedelta
TRAIN_PATH = Path("EDWaitTimes_fixed.xlsx")
TEST_PATH  = Path("EDWaitTimes.csv-test.xlsx")
DTR_PATH   = Path("dtr_a_18.xlsx")
OUT_PATH   = Path("submission.csv")

PC_RE = re.compile(
    r"\b([ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z])\s?(\d[ABCEGHJ-NPRSTV-Z]\d)\b",
    re.I
)

def normalize_name(name):
    s = "" if pd.isna(name) else str(name).strip()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"\.+$", "", s)
    return s

def extract_postal_code(address):
    if pd.isna(address):
        return np.nan
    m = PC_RE.search(str(address))
    return (m.group(1) + m.group(2)).upper() if m else np.nan

def extract_province(address):
    if pd.isna(address):
        return np.nan
    s = str(address).upper()
    m = re.search(r"\b(AB|BC|MB|NB|NL|NS|NT|NU|ON|PE|QC|SK|YT)\b", s)
    return m.group(1) if m else np.nan

def add_time_features(df):
    t = pd.to_datetime(df["time_dt"], errors="coerce")
    df["hour"] = t.dt.hour
    df["dow"] = t.dt.dayofweek
    df["month"] = t.dt.month
    df["is_weekend"] = (df["dow"] >= 5).astype(int)
    df["hour_bin3"] = (df["hour"] // 3).astype("Int64")  # 0..7
    return df

def smoothed_mean(df, group_cols, target, k=20.0):
    g = df[target].mean()
    stats = df.groupby(group_cols)[target].agg(["mean","count"])
    sm = (stats["count"]*stats["mean"] + k*g) / (stats["count"] + k)
    return sm, float(g)


# Holidays (BC specific, Apr–Aug) + Easter
def easter_sunday_gregorian(year: int) -> date:
    a = year % 19
    b = year // 100
    c = year % 100
    d = b // 4
    e = b % 4
    f = (b + 8) // 25
    g = (b - f + 1) // 3
    h = (19*a + b - d - g + 15) % 30
    i = c // 4
    k = c % 4
    l = (32 + 2*e + 2*i - h - k) % 7
    m = (a + 11*h + 22*l) // 451
    month = (h + l - 7*m + 114) // 31
    day = ((h + l - 7*m + 114) % 31) + 1
    return date(year, month, day)

def victoria_day(year: int) -> date:
    d = date(year, 5, 24)
    while d.weekday() != 0:
        d -= timedelta(days=1)
    return d

def bc_day(year: int) -> date:
    d = date(year, 8, 1)
    while d.weekday() != 0:
        d += timedelta(days=1)
    return d

def holiday_calendar(year: int) -> dict:
    hol = {}
    easter = easter_sunday_gregorian(year)
    hol[easter - timedelta(days=2)] = "Good Friday"
    hol[easter + timedelta(days=1)] = "Easter Monday"
    hol[victoria_day(year)] = "Victoria Day"

    canada = date(year, 7, 1)
    if canada.weekday() == 6:
        hol[canada + timedelta(days=1)] = "Canada Day (Obs)"
    elif canada.weekday() == 5:
        hol[canada + timedelta(days=2)] = "Canada Day (Obs)"
    else:
        hol[canada] = "Canada Day"

    hol[bc_day(year)] = "BC Day"
    return hol

def add_holiday_features(df: pd.DataFrame, time_col="time_dt") -> pd.DataFrame:
    out = df.copy()
    dt = pd.to_datetime(out[time_col], errors="coerce")
    out["date"] = dt.dt.date

    years = sorted({d.year for d in out["date"].dropna().unique()})
    hol_map = {}
    for y in years:
        hol_map.update(holiday_calendar(int(y)))
    hol_dates = set(hol_map.keys())

    out["holiday_name"] = out["date"].map(hol_map)
    out["is_holiday"] = out["holiday_name"].notna().astype(int)

    out["is_day_before_holiday"] = out["date"].map(
        lambda d: int(pd.notna(d) and ((d + timedelta(days=1)) in hol_dates))
    )
    out["is_day_after_holiday"] = out["date"].map(
        lambda d: int(pd.notna(d) and ((d - timedelta(days=1)) in hol_dates))
    )

    def is_long_weekend(d):
        if pd.isna(d):
            return 0
        monday = d - timedelta(days=d.weekday())
        if monday in hol_dates and monday.weekday() == 0:
            return int(d in {
                monday - timedelta(days=3),
                monday - timedelta(days=2),
                monday - timedelta(days=1),
                monday
            })
        return 0

    out["is_long_weekend"] = out["date"].map(is_long_weekend)
    out["holiday_name"] = out["holiday_name"].fillna("None").astype(str)
    return out


# train daily history features (hospital × date) 

def build_daily_history(train_df: pd.DataFrame):
    tmp = train_df[["name_norm", "time_dt", "waitTimeMinutes", "elosMinutes"]].copy()
    tmp["date"] = pd.to_datetime(tmp["time_dt"], errors="coerce").dt.date
    tmp["waitTimeMinutes"] = pd.to_numeric(tmp["waitTimeMinutes"], errors="coerce")
    tmp["elosMinutes"]     = pd.to_numeric(tmp["elosMinutes"], errors="coerce")

    daily = (
        tmp.dropna(subset=["name_norm", "date"])
           .groupby(["name_norm", "date"], as_index=False)
           .agg(
               daily_mean_wait=("waitTimeMinutes", "mean"),
               daily_mean_elos=("elosMinutes", "mean"),
               daily_count_wait=("waitTimeMinutes", "count"),
               daily_count_elos=("elosMinutes", "count"),
               daily_missing_wait=("waitTimeMinutes", lambda s: float(pd.to_numeric(s, errors="coerce").isna().mean())),
               daily_missing_elos=("elosMinutes",     lambda s: float(pd.to_numeric(s, errors="coerce").isna().mean())),
           )
    )

    daily = daily.sort_values(["name_norm", "date"])
    g = daily.groupby("name_norm", sort=False)

    # lag1
    for base in ["daily_mean_wait","daily_mean_elos","daily_count_wait","daily_count_elos","daily_missing_wait","daily_missing_elos"]:
        daily[f"{base}_lag1"] = g[base].shift(1)

    # rolling (past-only via shift(1))
    def roll_past(col, w):
        return g[col].shift(1).rolling(w).mean().reset_index(level=0, drop=True)

    for w in [3, 7]:
        daily[f"daily_mean_wait_roll{w}"] = roll_past("daily_mean_wait", w)
        daily[f"daily_mean_elos_roll{w}"] = roll_past("daily_mean_elos", w)
        daily[f"daily_count_wait_roll{w}"] = roll_past("daily_count_wait", w)
        daily[f"daily_count_elos_roll{w}"] = roll_past("daily_count_elos", w)
        daily[f"daily_missing_wait_roll{w}"] = roll_past("daily_missing_wait", w)
        daily[f"daily_missing_elos_roll{w}"] = roll_past("daily_missing_elos", w)

    hist_cols = [
        "daily_mean_wait_lag1","daily_mean_wait_roll3","daily_mean_wait_roll7",
        "daily_mean_elos_lag1","daily_mean_elos_roll3","daily_mean_elos_roll7",
        "daily_count_wait_lag1","daily_count_wait_roll3","daily_count_wait_roll7",
        "daily_count_elos_lag1","daily_count_elos_roll3","daily_count_elos_roll7",
        "daily_missing_wait_lag1","daily_missing_wait_roll3","daily_missing_wait_roll7",
        "daily_missing_elos_lag1","daily_missing_elos_roll3","daily_missing_elos_roll7",
    ]

    medians = {c: float(pd.to_numeric(daily[c], errors="coerce").median()) for c in hist_cols}
    # sensible fallbacks
    for c in hist_cols:
        if not np.isfinite(medians[c]):
            medians[c] = 0.0

    keep = ["name_norm", "date"] + hist_cols
    return daily[keep], medians

def merge_daily_history(base_df: pd.DataFrame, hist_daily: pd.DataFrame, medians: dict) -> pd.DataFrame:
    out = base_df.copy()
    if "date" not in out.columns:
        out["date"] = pd.to_datetime(out["time_dt"], errors="coerce").dt.date

    out = out.merge(hist_daily, on=["name_norm","date"], how="left")

    hist_cols = [c for c in hist_daily.columns if c not in ["name_norm","date"]]
    for c in hist_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(medians[c])

    # log1p for count histories
    for c in [x for x in hist_cols if "daily_count_" in x]:
        out[f"log1p_{c}"] = np.log1p(np.clip(pd.to_numeric(out[c], errors="coerce"), 0, None))

    return out

# per-hospital ASOF history features (time-level, past-only)
def build_asof_history(train_df: pd.DataFrame, y_col: str, prefix: str):
    """
    Creates a TRAIN-only history table with past-only features per hospital:
      {prefix}_last, {prefix}_roll10, {prefix}_roll30, {prefix}_delta_min
    """
    h = train_df[["name_norm", "time_dt", y_col]].copy()
    h["time_dt"] = pd.to_datetime(h["time_dt"], errors="coerce")
    h[y_col] = pd.to_numeric(h[y_col], errors="coerce")
    h = h.dropna(subset=["name_norm", "time_dt", y_col]).copy()

    h = h.sort_values(["name_norm", "time_dt"])
    g = h.groupby("name_norm", sort=False)

    shifted = g[y_col].shift(1)
    h[f"{prefix}_last"] = shifted
    h[f"{prefix}_roll10"] = shifted.rolling(10).mean().reset_index(level=0, drop=True)
    h[f"{prefix}_roll30"] = shifted.rolling(30).mean().reset_index(level=0, drop=True)

    prev_t = g["time_dt"].shift(1)
    h[f"{prefix}_delta_min"] = (h["time_dt"] - prev_t).dt.total_seconds() / 60.0

    feat_cols = [f"{prefix}_last", f"{prefix}_roll10", f"{prefix}_roll30", f"{prefix}_delta_min"]
    hist = h[["name_norm", "time_dt"] + feat_cols].copy()

    med = {c: float(pd.to_numeric(hist[c], errors="coerce").median()) for c in feat_cols}
    for c in feat_cols:
        if not np.isfinite(med[c]):
            med[c] = 0.0

    return hist, med, feat_cols

def merge_asof_features(df: pd.DataFrame, hist: pd.DataFrame, feat_cols, medians: dict):
    """
    Merge per-hospital history into df using merge_asof (past-only).
    Requirements: both sides sorted by time_dt (primary) and name_norm for by merge.
    """
    out = df.copy()
    out["time_dt"] = pd.to_datetime(out["time_dt"], errors="coerce")

    # init cols
    for c in feat_cols:
        out[c] = np.nan

    left = out[["name_norm", "time_dt"]].dropna(subset=["name_norm", "time_dt"]).copy()
    left = left.sort_values(["time_dt", "name_norm"]).reset_index()  # keep original index

    right = hist.dropna(subset=["name_norm", "time_dt"]).copy()
    right = right.sort_values(["time_dt", "name_norm"]).reset_index(drop=True)

    merged = pd.merge_asof(
        left,
        right,
        by="name_norm",
        on="time_dt",
        direction="backward",
        allow_exact_matches=True,
    )

    out.loc[merged["index"].values, feat_cols] = merged[feat_cols].values

    for c in feat_cols:
        out[c] = pd.to_numeric(out[c], errors="coerce").fillna(medians[c])
        out[f"log1p_{c}"] = np.log1p(np.clip(out[c], 0, None))

    return out
train_raw = pd.read_excel(TRAIN_PATH)
test_raw  = pd.read_excel(TEST_PATH)

# Preserve exact original strings for submission
test_raw["name_orig"] = test_raw["name"]
test_raw["time_orig"] = test_raw["time"]

train_raw["time_dt"] = pd.to_datetime(train_raw["time"], errors="coerce")
test_raw["time_dt"]  = pd.to_datetime(test_raw["time"], errors="coerce")

train_raw["name"] = train_raw["name"].astype(str).str.strip()
test_raw["name"]  = test_raw["name"].astype(str).str.strip()

train_raw["name_norm"] = train_raw["name"].map(normalize_name)
test_raw["name_norm"]  = test_raw["name"].map(normalize_name)


# Hospital lookup (train only)

meta = train_raw[["name","address","latitude","longitude","open247"]].drop_duplicates("name").copy()
meta["name_norm"]  = meta["name"].map(normalize_name)
meta["postalcode"] = meta["address"].map(extract_postal_code)
meta["province"]   = meta["address"].map(extract_province)

hospital_lookup = (
    meta.groupby("name_norm", as_index=False)
        .agg({
            "latitude": "median",
            "longitude": "median",
            "open247": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
            "postalcode": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
            "province": lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan,
        })
)

# DTR + FSA fallback
dtr = pd.read_excel(DTR_PATH)
dtr["postalcode18"] = dtr["postalcode18"].astype(str).str.upper().str.replace(" ", "", regex=False)

road_cols = [c for c in dtr.columns if str(c).lower().startswith("dtr")]
for c in road_cols:
    dtr[c] = pd.to_numeric(dtr[c], errors="coerce")
    dtr.loc[dtr[c].isin([-9999, -1111]), c] = np.nan

dtr["fsa"] = dtr["postalcode18"].str[:3]
dtr_fsa = dtr.groupby("fsa")[road_cols].mean().reset_index()
dtr_fsa.columns = ["fsa"] + [f"{c}_fsa_mean" for c in road_cols]
road_medians = {c: float(dtr[c].median()) for c in road_cols}


# Base feature builder
train_start = pd.to_datetime(train_raw["time_dt"], errors="coerce").min()

def build_base(df):
    X = df[["name_norm","time_dt"]].copy()
    X = X.merge(hospital_lookup, on="name_norm", how="left")

    # holidays
    X = add_holiday_features(X, time_col="time_dt")

    # DTR join
    X["fsa"] = X["postalcode"].astype(str).str[:3].replace("nan","UNK")
    X = X.merge(dtr[["postalcode18"] + road_cols], left_on="postalcode", right_on="postalcode18", how="left")
    X = X.merge(dtr_fsa, on="fsa", how="left")

    for c in road_cols:
        X[c] = X[c].fillna(X[f"{c}_fsa_mean"]).fillna(road_medians[c])
        X[f"log1p_{c}"] = np.log1p(X[c])

    # time features
    X = add_time_features(X)
    X["days_since_start"] = (
        (pd.to_datetime(X["time_dt"], errors="coerce") - train_start).dt.total_seconds()/(3600*24)
    )

    # cats
    X["open247_cat"] = X["open247"].map(lambda v: "True" if v is True else ("False" if v is False else "Unknown"))
    X["province_cat"] = X["province"].fillna("UNK")

    # numeric cast (include holiday numeric flags)
    holiday_num = ["is_holiday","is_day_before_holiday","is_day_after_holiday","is_long_weekend"]
    num_cols = ["latitude","longitude","hour","dow","month","is_weekend","days_since_start"] + holiday_num + road_cols + [f"log1p_{c}" for c in road_cols]
    for c in num_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    return X

train_base = build_base(train_raw)
test_base  = build_base(test_raw)

# Daily history features 

hist_daily, hist_medians = build_daily_history(train_raw)
train_base = merge_daily_history(train_base, hist_daily, hist_medians)
test_base  = merge_daily_history(test_base,  hist_daily, hist_medians)

daily_hist_cols = [c for c in hist_daily.columns if c not in ["name_norm","date"]]
daily_hist_cols += [c for c in train_base.columns if c.startswith("log1p_daily_count_")]

# ASOF time-level history features 
hist_wait_asof, med_wait_asof, wait_asof_cols = build_asof_history(train_raw, "waitTimeMinutes", "wait_asof")
hist_elos_asof, med_elos_asof, elos_asof_cols = build_asof_history(train_raw, "elosMinutes", "elos_asof")

train_base = merge_asof_features(train_base, hist_wait_asof, wait_asof_cols, med_wait_asof)
train_base = merge_asof_features(train_base, hist_elos_asof, elos_asof_cols, med_elos_asof)
test_base  = merge_asof_features(test_base,  hist_wait_asof, wait_asof_cols, med_wait_asof)
test_base  = merge_asof_features(test_base,  hist_elos_asof, elos_asof_cols, med_elos_asof)

asof_cols = wait_asof_cols + elos_asof_cols
asof_cols += [f"log1p_{c}" for c in wait_asof_cols] + [f"log1p_{c}" for c in elos_asof_cols]

# Target encodings (Train only), then apply to both
train_te_src = train_base.copy()
train_te_src["waitTimeMinutes"] = pd.to_numeric(train_raw["waitTimeMinutes"], errors="coerce")
train_te_src["elosMinutes"]     = pd.to_numeric(train_raw["elosMinutes"], errors="coerce")

# Base TE
h_wait, g_wait = smoothed_mean(train_te_src.dropna(subset=["waitTimeMinutes"]), ["name_norm"], "waitTimeMinutes")
hh_wait, _     = smoothed_mean(train_te_src.dropna(subset=["waitTimeMinutes"]), ["name_norm","hour"], "waitTimeMinutes")
hb_wait, _     = smoothed_mean(train_te_src.dropna(subset=["waitTimeMinutes"]), ["name_norm","hour_bin3"], "waitTimeMinutes")
hd_wait, _     = smoothed_mean(train_te_src.dropna(subset=["waitTimeMinutes"]), ["name_norm","dow"], "waitTimeMinutes")

h_elos, g_elos = smoothed_mean(train_te_src.dropna(subset=["elosMinutes"]), ["name_norm"], "elosMinutes")
hh_elos, _     = smoothed_mean(train_te_src.dropna(subset=["elosMinutes"]), ["name_norm","hour"], "elosMinutes")
hb_elos, _     = smoothed_mean(train_te_src.dropna(subset=["elosMinutes"]), ["name_norm","hour_bin3"], "elosMinutes")
hd_elos, _     = smoothed_mean(train_te_src.dropna(subset=["elosMinutes"]), ["name_norm","dow"], "elosMinutes")

def apply_te(base_df):
    out = base_df.copy()

    # hospital
    hosp_key = pd.MultiIndex.from_frame(out[["name_norm"]])
    out["te_hosp_wait"] = pd.Series(hosp_key.map(h_wait), index=out.index).astype(float).fillna(g_wait)
    out["te_hosp_elos"] = pd.Series(hosp_key.map(h_elos), index=out.index).astype(float).fillna(g_elos)

    # hospital-hour
    hh_key = pd.MultiIndex.from_frame(out[["name_norm","hour"]])
    te_hw = pd.Series(hh_key.map(hh_wait), index=out.index).astype(float)
    te_he = pd.Series(hh_key.map(hh_elos), index=out.index).astype(float)
    out["te_hosp_hour_wait"] = te_hw.where(te_hw.notna(), out["te_hosp_wait"])
    out["te_hosp_hour_elos"] = te_he.where(te_he.notna(), out["te_hosp_elos"])

    # hospital-hourbin3
    hb_key = pd.MultiIndex.from_frame(out[["name_norm","hour_bin3"]])
    te_hbw = pd.Series(hb_key.map(hb_wait), index=out.index).astype(float)
    te_hbe = pd.Series(hb_key.map(hb_elos), index=out.index).astype(float)
    out["te_hosp_hbin_wait"] = te_hbw.where(te_hbw.notna(), out["te_hosp_wait"])
    out["te_hosp_hbin_elos"] = te_hbe.where(te_hbe.notna(), out["te_hosp_elos"])

    # hospital-dow
    hd_key = pd.MultiIndex.from_frame(out[["name_norm","dow"]])
    te_hdw = pd.Series(hd_key.map(hd_wait), index=out.index).astype(float)
    te_hde = pd.Series(hd_key.map(hd_elos), index=out.index).astype(float)
    out["te_hosp_dow_wait"] = te_hdw.where(te_hdw.notna(), out["te_hosp_wait"])
    out["te_hosp_dow_elos"] = te_hde.where(te_hde.notna(), out["te_hosp_elos"])

    return out

train_feat = apply_te(train_base)
test_feat  = apply_te(test_base)

# Categorical features
cat_cols = ["fsa","open247_cat","province_cat","holiday_name"]
for c in cat_cols:
    train_feat[c] = train_feat[c].fillna("UNK").astype("category")
    test_feat[c]  = test_feat[c].fillna("UNK").astype("category")

holiday_num = ["is_holiday","is_day_before_holiday","is_day_after_holiday","is_long_weekend"]

# Feature list
te_cols = [
    "te_hosp_wait","te_hosp_elos",
    "te_hosp_hour_wait","te_hosp_hour_elos",
    "te_hosp_hbin_wait","te_hosp_hbin_elos",
    "te_hosp_dow_wait","te_hosp_dow_elos",
]

feature_cols = [
    "latitude","longitude","hour","dow","month","is_weekend","hour_bin3","days_since_start",
] + holiday_num + te_cols + road_cols + [f"log1p_{c}" for c in road_cols] + daily_hist_cols + asof_cols + cat_cols

X_train = train_feat[feature_cols].copy()
X_test  = test_feat[feature_cols].copy()

def fit_lgb(y, objective="quantile", alpha=0.4):
    y = pd.to_numeric(y, errors="coerce")
    mask = y.notna()
    X = X_train.loc[mask]
    y = y.loc[mask]

    params = dict(
        n_estimators=9000,
        learning_rate=0.02,
        num_leaves=192,
        min_data_in_leaf=50,
        feature_fraction=0.85,
        bagging_fraction=0.85,
        bagging_freq=1,
        lambda_l2=1.0,
        random_state=42,
        n_jobs=-1,
    )

    if objective == "quantile":
        model = lgb.LGBMRegressor(objective="quantile", alpha=alpha, **params)
    elif objective == "regression_l1":
        model = lgb.LGBMRegressor(objective="regression_l1", **params)
    else:
        model = lgb.LGBMRegressor(objective=objective, **params)

    model.fit(X, np.log1p(y), categorical_feature=cat_cols)
    return model

WQ = 0.7
WL = 0.3

# Wait time
m_wait_q = fit_lgb(train_raw["waitTimeMinutes"], objective="quantile", alpha=0.4)
m_wait_l = fit_lgb(train_raw["waitTimeMinutes"], objective="regression_l1")
pred_wait = WQ * m_wait_q.predict(X_test) + WL * m_wait_l.predict(X_test)
pred_wait = np.clip(np.expm1(pred_wait), 0, None)

# ELOS
m_elos_q = fit_lgb(train_raw["elosMinutes"], objective="quantile", alpha=0.4)
m_elos_l = fit_lgb(train_raw["elosMinutes"], objective="regression_l1")
pred_elos = WQ * m_elos_q.predict(X_test) + WL * m_elos_l.predict(X_test)
pred_elos = np.clip(np.expm1(pred_elos), 0, None)

submission = pd.DataFrame({
    "name": test_raw["name_orig"],
    "time": test_raw["time_orig"],
    "waitTimeMinutes": pred_wait,
    "elosMinutes": pred_elos,
})
submission.to_csv(OUT_PATH, index=False)
print("Wrote submission.csv", submission.shape)
print(submission.head())


ModuleNotFoundError: No module named 'lightgbm'